# 02 · Box Least Squares, step by step

From a detrended light curve to a vetted transit candidate: the period grid, the periodogram, folding, the trapezoid fit, vetting, iterative multi-planet search, and an eclipsing binary caught by the odd/even test.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from transit_hunter.config import load_data_config, setup_logging

setup_logging("WARNING")
cfg = load_data_config()
plt.rcParams.update({"figure.figsize": (12, 3.5), "axes.grid": True, "grid.alpha": 0.3})
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#199e70"

In [ ]:
TARGET = "Kepler-10"
MISSION = None
MAX_SEGMENTS = 4

In [ ]:
from transit_hunter.data.fetch import fetch_lightcurve
from transit_hunter.data.synthetic import synthetic_planet_lightcurve

try:
    lc = fetch_lightcurve(TARGET, mission=MISSION, max_segments=MAX_SEGMENTS)
    time, flux, flux_err = lc.time, lc.flux, lc.flux_err
    label = f"{lc.target_id} ({lc.mission}, {lc.author}, segments {lc.segments})"
except Exception as exc:  # offline, MAST down, ...
    print(f"Could not fetch {TARGET!r} ({type(exc).__name__}: {exc}).\nUsing a synthetic Kepler-10b-like light curve instead.")
    syn = synthetic_planet_lightcurve(period=0.8375, t0=0.3, depth=0.00025, duration=0.075,
                                      baseline_days=120, noise_ppm=80, variability_ppm=600, seed=1)
    time, flux, flux_err = syn.time, syn.flux, syn.flux_err
    label = "synthetic Kepler-10b-like star"
print(label, f"| {time.size:,} points over {time.max() - time.min():.1f} days")

In [ ]:
from transit_hunter.processing.clean import clean_lightcurve
from transit_hunter.processing.detrend import detrend

c = clean_lightcurve(time, flux, flux_err, cfg.cleaning)
d = detrend(c.time, c.flux, c.flux_err, cfg.detrend)

## 1. The period grid

Neighboring trial periods must be close enough that a transit can't drift by more than a fraction of its duration over the whole baseline. Longer baselines therefore need many more trial periods.

In [ ]:
from transit_hunter.processing.search import period_grid

baseline = d.time.max() - d.time.min()
grid = period_grid(baseline, cfg.search)
print(f"baseline {baseline:.1f} d -> {grid.size:,} trial periods from {grid[0]:.2f} to {grid[-1]:.1f} d")
for b in (27, 90, 365, 1460):
    print(f"  baseline {b:5d} d: {period_grid(b, cfg.search).size:>8,} periods")

## 2. Run BLS and look at the periodogram

In [ ]:
from transit_hunter.processing.search import bls_search

cand = bls_search(d.time, d.flux, d.flux_err, cfg.search)
print(f"P = {cand.period:.5f} d, duration = {cand.duration * 24:.2f} h, depth = {cand.depth * 1e6:.0f} ppm, "
      f"SNR = {cand.snr:.1f}, SDE = {cand.sde:.1f}, transits = {cand.n_transits}, significant = {cand.significant}, "
      f"{cand.search_seconds:.1f} s")

p, s = np.array(cand.periodogram.period), np.array(cand.periodogram.power)
plt.semilogx(p, s, color=BLUE, lw=0.8)
for mult, lab in ((0.5, "P/2"), (2, "2P")):
    plt.axvline(cand.period * mult, color="0.5", ls=":", lw=1); plt.text(cand.period * mult, s.max(), lab, color="0.4")
plt.plot(cand.period, s[np.argmin(abs(p - cand.period))], "o", color=ORANGE, ms=8)
plt.xlabel("Trial period (days)"); plt.ylabel("SNR"); plt.title("BLS periodogram"); plt.show()

## 3. Fold, fit and vet

In [ ]:
from transit_hunter.pipeline import fit_trapezoid
from transit_hunter.processing.fold import phase_fold
from transit_hunter.data.synthetic import trapezoid_transit
from transit_hunter.processing.features import build_example

ph = phase_fold(d.time, cand.period, cand.t0)
fit = fit_trapezoid(ph, d.flux, cand.duration, cand.depth)
print(f"BLS duration {cand.duration * 24:.2f} h -> trapezoid T14 {fit['duration'] * 24:.2f} h, "
      f"ingress fraction {fit['ingress_fraction']:.2f} (0.5 = V-shaped)")
near = np.abs(ph) < 3 * fit["duration"]
x = np.linspace(-3 * fit["duration"], 3 * fit["duration"], 400)
plt.plot(ph[near] * 24, (d.flux[near] - 1) * 1e6, ".", ms=2, color="0.6")
plt.plot(x * 24, (trapezoid_transit(x, 1e6, fit["t_offset"], fit["duration"], fit["depth"], fit["ingress_fraction"]) - 1) * 1e6, color=ORANGE, lw=2)
plt.xlabel("Hours from mid-transit"); plt.ylabel("ppm"); plt.title("Folded transit + trapezoid fit"); plt.show()

ex = build_example(d.time, d.flux, cand.period, cand.t0 + fit["t_offset"], fit["duration"], cfg.views, stellar_radius_rsun=1.06)
for f in ex.vetting.flags:
    print(f"{'PASS' if f.passed else ('FLAG' if f.passed is False else ' n/a')}  {f.name:<20} {f.message}")

## 4. Iterative search: more than one planet

Mask the first signal's transits, then search again. On real Kepler-10 data with enough quarters, the second signal is Kepler-10 c (about 45 d).

In [ ]:
from transit_hunter.processing.search import iterative_search

for k in iterative_search(d.time, d.flux, d.flux_err, cfg.search, max_candidates=3):
    print(f"#{k.rank}: P = {k.period:9.4f} d  depth = {k.depth * 1e6:7.0f} ppm  SNR = {k.snr:6.1f}  significant = {k.significant}")

## 5. An eclipsing binary at half its period

Two near-equal eclipses per orbit make BLS lock onto P/2. Then every other "transit" is the secondary eclipse, and the odd/even depths differ.

In [ ]:
from transit_hunter.data.synthetic import synthetic_eb_lightcurve

eb = synthetic_eb_lightcurve(period=2.2, secondary_ratio=0.75, seed=4)
ce = clean_lightcurve(eb.time, eb.flux, eb.flux_err, cfg.cleaning)
de = detrend(ce.time, ce.flux, ce.flux_err, cfg.detrend)
k = bls_search(de.time, de.flux, de.flux_err, cfg.search)
print(f"true period 2.2 d, BLS found {k.period:.4f} d")
ex_eb = build_example(de.time, de.flux, k.period, k.t0, max(k.duration, 0.1), cfg.views)
v = ex_eb.vetting
print(f"odd depth {v.depth_odd * 1e6:.0f} ppm vs even depth {v.depth_even * 1e6:.0f} ppm -> {v.odd_even_sigma:.0f} sigma apart")
ph = phase_fold(de.time, 2 * k.period, k.t0)
plt.plot(ph, (de.flux - 1) * 100, ".", ms=2, color=BLUE)
plt.xlabel("Days from primary eclipse (folded at 2 x BLS period)"); plt.ylabel("%"); plt.title("Folded at the true period: two different eclipses"); plt.show()

## 6. Engines agree (and how fast they are)

In [ ]:
import time as _t
for engine in ("numba", "astropy"):
    sc = cfg.search.model_copy(update={"engine": engine})
    t0 = _t.perf_counter(); r = bls_search(d.time[:20000], d.flux[:20000], d.flux_err[:20000], sc, keep_periodogram=False)
    print(f"{engine:>8}: P = {r.period:.5f} d, depth = {r.depth * 1e6:.0f} ppm, {_t.perf_counter() - t0:.1f} s")